# STAC Client Implementation Outline

## Client Search Scenario

Connected Data Asset STAC clients are, at their core, just conventional STAC clients respecting a number of Best Practices to make the STAC API protocol useful in the WGISS environment. The collection identifier of the collection of interest is a mandatory element in a CEOS STAC request to find granules (a.k.a. STAC items). Clients could retrieve collection IDs from the IDN or FedEO Collection search responses.

This chapter will give brief steps about how to retrieve a collection ID and how to interact with the granule search server for inventory search. The corresponding details are elaborated in the Use Case chapters.

We assume that Data providers have registered the metadata of their archived collections into the IDN or have made their collections accessible to FedEO. The client can query the IDN or FedEO to retrieve the collection ID for a desired collection of interest and, based on that collection ID and other spatial-temporal query conditions, build a valid CEOS STAC query. The following steps describe the client search scenario starting with a STAC Collection search.

In [328]:
%pip install geopandas
%pip install pystac_client==0.8.5

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [329]:
%pip install folium matplotlib mapclassify
%pip install jsonpath_ng

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [330]:
import folium
import folium.plugins
import geopandas as gpd
import shapely.geometry
import pandas as pd  
import numpy as np
import json
import urllib.parse
import requests

from xml.dom import minidom
from IPython.display import HTML, display
from IPython.display import Markdown as md
from pystac_client import Client
from pystac import Collection
from typing import Any, Dict
from urllib.parse import urlparse, parse_qsl
from matplotlib import pyplot as plt, cm, colors
from PIL import Image
from io import BytesIO
from branca.element import Figure
from concurrent.futures import ThreadPoolExecutor

import warnings
#warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', category=UserWarning)


def convert_bounds(bbox, invert_y=False):
    """
    Helper method for changing bounding box representation to leaflet notation

    ``(lon1, lat1, lon2, lat2) -> ((lat1, lon1), (lat2, lon2))``
    """
    x1, y1, x2, y2 = bbox
    if invert_y:
        y1, y2 = y2, y1
    return ((y1, x1), (y2, x2))

def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False
    

def curl_command( url: str, method: str = "GET" ) -> str:
    """
    Convert request URL to equivalent curl GET or POST command-line
    for STAC search (bash shell).
    """
    c = "curl -X " + method
    res = urlparse(url)
    if "GET" in method:
        c = c + " -G " + res.scheme + "://" + res.netloc + res.path
    else:
        c = c + " " + res.scheme + "://" + res.netloc + res.path    \
              + " \\\n\t--header 'Content-Type: application/json'"  \
              + " \\\n\t--data-raw '{"
    
    lst = parse_qsl(res.query)
    
    first = True 
    for i in lst:
        # print(i[0])
        # add \ to end of previous line
        if "GET" in method:
            # correction 16/3: data-urlencode used.
            v = i[1].replace('"','\\"')
            c = c + ' \\\n\t--data-urlencode "'+i[0]+'='+v+'"'
        else:
            if not(first):
                c = c + ','
                
            if i[0] in ["ids","collections"]:
                # "collections" and "ids") parameter has to be included as an array.
                lst = i[1].split(',')       
                c = c + '\n\t\t"'+i[0]+'": '+str(lst).replace("'","\"")
            
            elif is_number(i[1]) or i[1][0]=='{' or i[1][0]=='[':
                # do not surround with quotes if numerical value  or a json object or an array
                c = c + '\n\t\t"'+i[0]+'": '+i[1]
            else:
                c = c + '\n\t\t"'+i[0]+'": "'+i[1]+'"'
        first = False
        
    if "POST" in method:
        c = c + "\n\t}'"
    return c


def display_previews(results):
    """
    Helper method for displaying a grid of quicklooks (if available)
    """
    # create figure
    fig = plt.figure(figsize=(20, 20))
  
    # setting values to rows and column variables for the image grid
    rows = 8
    columns = 2
    pos = 1

    for item in results.items():
        # print(item.id)
        assets = item.assets
        try:
            # print("found thumbnail", assets['thumbnail'].href)
            url = assets['thumbnail'].href
            response = requests.get(url)
            Image1 = Image.open(BytesIO(response.content))
            # display at position 'pos'
            fig.add_subplot(rows, columns, pos)
            pos = pos+1
            # show the image
            plt.imshow(Image1)
            plt.axis('off')
            plt.title(item.id)
        except:
            pass
    return

def display_gdf_plot(results):
    """
    Helper method for displaying results as dataframe plot
    """
    # https://github.com/opendatacube/odc-stac (License Apache 2.0)
    # https://odc-stac.readthedocs.io/en/latest/notebooks/stac-load-e84-aws.html#Plot-STAC-Items-on-a-Map

    # Convert STAC items into a GeoJSON FeatureCollection
    stac_json = results.item_collection_as_dict()

    gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")

    fig = gdf.plot(
        "datetime",
        edgecolor="black",
        categorical=True,
        aspect="equal",
        alpha=0.5,
        figsize=(6, 12),
        legend=True,
        legend_kwds={"loc": "upper left", "frameon": False, "ncol": 1},
    )
    _ = fig.set_title("STAC Query Results")

    # gdf
    return

def display_date_distribution(results):
    """
    Helper method for displaying number of results per year/month as bar chart
    """
    items = list(results.items())
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json)

    gdf['date'] = pd.to_datetime(gdf['start_datetime'])
    # create a representation of the month with strfmt
    gdf['year_month'] = gdf['date'].map(lambda dt: dt.strftime('%Y-%m'))
    grouped_df = gdf.groupby('year_month')['year_month'].size().to_frame("count").reset_index()
    grouped_df.plot(kind='bar', x='year_month', y='count')
    return

def display_value_distribution(results, column):
    """
    Helper method for displaying number values in column as bar chart
    """
    items = list(results.items())
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json)

    # gdf['date'] = pd.to_datetime(gdf['start_datetime'])
    # create a representation of the month with strfmt
    # gdf['year_month'] = gdf['date'].map(lambda dt: dt.strftime('%Y-%m'))

    try:
        grouped_df = gdf.groupby(column)[column].size().to_frame("count").reset_index()
        grouped_df.plot(kind='bar', x=column, y='count')
    except:
        print(column + " values are not available.")

    return


def display_map(results):
    """
    Helper method for displaying results on a map
    """
    # https://github.com/python-visualization/folium/issues/1501
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")
    
    fig = Figure(width="800px", height="500px")
    map1 = folium.Map()
    fig.add_child(map1)

    # folium.GeoJson(
    #    shapely.geometry.box(*bbox),  # ??
    #    style_function=lambda x: dict(fill=False, weight=1, opacity=0.7, color="olive"),
    #    name="Query",
    # ).add_to(map1)

    gdf.explore(
        "start_datetime",
        categorical=True,
        tooltip=[
            "title", "datetime", "start_datetime", "platform", "instruments"    
        ],
        popup=True,
        style_kwds=dict(fillOpacity=0.1, width=2),
        name="STAC",
        m=map1,
    )

    # map1.fit_bounds(bounds=convert_bounds(gdf.unary_union.bounds))
    map1.fit_bounds(bounds=convert_bounds(gdf.union_all().bounds))
    display(fig)
    return


In [331]:
# URL_LANDING_PAGE =  'https://fedeo.ceos.org/' 
URL_LANDING_PAGE =  'https://geo.spacebel.be/' 

In [332]:
COLLECTION_ID1 = 'SeaSat.ESA.archive'
COLLECTION_ID2 = 'LANDSAT.TM.GTC'

**Step 1**  
>  Obtain the STAC API Collection search endpoint to formulate a valid Collection search request.

In [333]:
md(f"The API implements the STAC API Collection Search Extension [[RD25]](#RD25).  The URL of the collection search endpoint is advertized in the Landing Page `{URL_LANDING_PAGE}` as a link (rel='data'). The code below extracts this link from the landing page.")

The API implements the STAC API Collection Search Extension [[RD25]](#RD25).  The URL of the collection search endpoint is advertized in the Landing Page `https://geo.spacebel.be/` as a link (rel='data'). The code below extracts this link from the landing page.

In [334]:
from jsonpath_ng.ext import parse

response = requests.get(URL_LANDING_PAGE)
data = json.loads(response.text)
# Extract ´data´ link
expression = parse("$.links[?(@.rel == 'data')].href")
r = expression.find(data)
r[0].value

'https://geo.spacebel.be/collections'

The link with rel="http://www.opengis.net/def/rel/ogc/1.0/queryables" in the collection search response provides access to the list of filter criteria available for collection search.  It returns a Queryables object in JSON Schema format.
The list of filter criteria IDN and FedEO support for collection searches differs.

In [335]:
# Retrieve /collections response
response = requests.get(r[0].value)
data = json.loads(response.text)

In [336]:
# Extract ´queryables´ link
expression = parse("$.links[?(@.rel == 'http://www.opengis.net/def/rel/ogc/1.0/queryables')].href")
r = expression.find(data)
r[0].value

'https://geo.spacebel.be/collections/queryables'

In [337]:
# Get queryables response and list supported filter parameters alphabetically.
response = requests.get(r[0].value)
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

,key,type,format
15,accessConstraint,string,-
6,ceosard:specification,string,-
12,ceosard:specification_version,string,-
19,classifiedAs,string,uri
5,collection,string,-
10,datetime,string,-
3,description,string,-
4,externalId,string,-
2,format,string,-
16,id,string,-


**Step 2**  
>  Search collections of interest through STAC with proper request parameters (e.g. spatial footprint, temporal extent or keyword).

The collection search supports free text searches using the `q` query parameter as shown below.

In [338]:
from pystac_client import Client
api = Client.open(URL_LANDING_PAGE)

results = api.collection_search(
    q = 'Seasat'
)

In [339]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://geo.spacebel.be/collections \
	--data-urlencode "q=Seasat"
```


In [340]:
data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=['collections'])
df[['id', 'keywords']]

,id,keywords
0,SeaSat.ESA.archive,"[Oceans, EARTH SCIENCE > OCEANS, Ocean Circula..."


**Step 3**  
>  From the collection search response, obtain the granule search endpoint for this collection (rel="items"), or use the collection ID to be used in a cross-collection search request (rel="search" in Landing page).

The cross collection granule search endpoint is advertized in the Landing Page as a link with rel="search".

In [341]:
# Extract the link with rel="search" and method="GET"
response = requests.get(URL_LANDING_PAGE)
data = json.loads(response.text)
expression = parse("$.links[?(@.method == 'GET' & @.rel == 'search' )].href")
r = expression.find(data)
r[0].value

'https://geo.spacebel.be/search'

The granule search endpoint for the individual collection is advertized in the collection metadata as a link with rel="items".  The collection metadata can be found inside a collection search response or be accessed directly if its collection ID is known.  The server may support both the `GET` and `POST` methods.

In [342]:
md(f"For example, the collection metadata for `{COLLECTION_ID1}`, is available \
at {URL_LANDING_PAGE + 'collections/' + COLLECTION_ID1}.  This corresponds to one of the many representations available using content-negotiation.")

For example, the collection metadata for `SeaSat.ESA.archive`, is available at https://geo.spacebel.be/collections/SeaSat.ESA.archive.  This corresponds to one of the many representations available using content-negotiation.

In [343]:
URL = URL_LANDING_PAGE + 'collections/' + COLLECTION_ID1

In [344]:
curl_str = curl_command(URL)
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://geo.spacebel.be/collections/SeaSat.ESA.archive
```


In [345]:
md(f"Let's take an other collection `{COLLECTION_ID2}` from an optical mission.")

Let's take an other collection `LANDSAT.TM.GTC` from an optical mission.

In [346]:
URL = URL_LANDING_PAGE + 'collections/' + COLLECTION_ID2

In [347]:
curl_str = curl_command(URL)
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://geo.spacebel.be/collections/LANDSAT.TM.GTC
```


In [348]:
# retrieve collection by identifier and extract granule search endpoint
response = requests.get(URL)
data_collections = json.loads(response.text)

expression = parse("$.links[?(@.rel == 'items' )].href")
r = expression.find(data_collections)
r[0].value

'https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items'

**Step 4**  
>  Based on the common query parameters or the `filter` query parameter using supported filter parameters found in Queryables, formulate a STAC API search request for granules belonging to that collection, directed to the cross-collection search endpoint (rel="search").

In [349]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',         
    max_items = 2,
    collections=[COLLECTION_ID2],
    bbox = [14.90, 37.700, 14.99, 37.780], # Mount Etna
    datetime=['2010-01-01T00:00:00Z', '2011-01-02T00:00:00Z'],
    limit = 2
)

granule_results = results # keep for further use below.

In [350]:
curl_str = curl_command(results.url_with_parameters(), "GET")
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://geo.spacebel.be/search \
	--data-urlencode "limit=2" \
	--data-urlencode "bbox=14.9,37.7,14.99,37.78" \
	--data-urlencode "datetime=2010-01-01T00:00:00Z/2011-01-02T00:00:00Z" \
	--data-urlencode "collections=LANDSAT.TM.GTC"
```


In [351]:
print(f"{results.matched()} items found.")

31 items found.


In [352]:
# Show search response (GeoJSON)
data = results.item_collection_as_dict()
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

```json
{
   "type": "FeatureCollection",
   "features": [
      {
         "stac_version": "1.0.0",
         "assets": {
            "enclosure_zip": {
               "roles": [
                  "data"
               ],
               "href": "https://landsat-diss.eo.esa.int/oads/data/LandsatTM/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP",
               "published": "2010-12-28T09:32:21Z",
               "type": "application/zip",
               "title": "Download"
            },
            "thumbnail": {
               "roles": [
                  "thumbnail"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/thumbnail/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_TIMG.jpg",
               "type": "image/jpeg",
               "title": "THUMBNAIL"
            },
            "offering_2": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetTile"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "title": "GetTile",
               "type": "image/jpgpng"
            },
            "quicklook_png": {
               "roles": [
                  "overview"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/browse/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_BID.PNG",
               "type": "image/png",
               "title": "QUICKLOOK"
            },
            "offering_1": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml",
               "title": "GetCapabilities",
               "type": "application/xml"
            },
            "metadata_ogc_10_157r4": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/gml%2Bxml&recordSchema=om",
               "title": "OGC 10-157r4 metadata",
               "type": "application/gml+xml;profile=\"http://www.opengis.net/spec/EOMPOM/1.1\""
            },
            "offering_4": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetMap"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetMap&layers=Landsat-TM___heatmap&format=image/png&TRANSPARENT=true&TIME=2010-12-28T09:31:52Z/2010-12-28T09:32:21Z&width={Width}&height={Height}&SRS=EPSG:4326&styles={Styles}&bbox={BoundingBox}&dim_range={DimRange}",
               "title": "GetMap",
               "type": "image/png"
            },
            "metadata_ogc_17_003r2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?mode=owc",
               "title": "OGC 17-003r2 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eo-geojson/1.0\""
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "offering_3": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetCapabilities",
               "title": "GetCapabilities",
               "type": "application/xml"
            }
         },
         "bbox": [
            12.5999,
            36.5266,
            15.1541,
            38.4181
         ],
         "geometry": {
            "coordinates": [
               [
                  [
                     13.0533,
                     38.4181
                  ],
                  [
                     15.1541,
                     38.1131
                  ],
                  [
                     14.6573,
                     36.5266
                  ],
                  [
                     12.5999,
                     36.8248
                  ],
                  [
                     13.0533,
                     38.4181
                  ]
               ]
            ],
            "type": "Polygon"
         },
         "links": [
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "wms:transparent": true,
               "rel": "wms",
               "wms:dimensions": {
                  "dim_range": "{DimRange}",
                  "TIME": "2010-12-28T09:31:52Z/2010-12-28T09:32:21Z",
                  "version": "1.1.1"
               },
               "href": "https://vs.eo.esa.int/wms07/ows",
               "type": "image/png",
               "title": "LANDSAT.TM.GTC product",
               "wms:layers": [
                  "Landsat-TM___heatmap"
               ],
               "wms:styles": [
                  "{Styles}"
               ]
            },
            {
               "wmts:encoding": "rest",
               "uriTemplate": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "rel": "wmts",
               "wmts:layer": "Landsat-TM",
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml"
            },
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "canonical",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "collection",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
         "collection": "LANDSAT.TM.GTC",
         "type": "Feature",
         "stac_extensions": [
            "https://stac-extensions.github.io/eo/v2.0.0/schema.json",
            "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
            "https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json",
            "https://stac-extensions.github.io/sat/v1.2.0/schema.json",
            "https://stac-extensions.github.io/view/v1.0.0/schema.json",
            "https://stac-extensions.github.io/product/v1.0.0/schema.json",
            "https://stac-extensions.github.io/grid/v1.1.0/schema.json",
            "https://stac-extensions.github.io/version/v1.2.0/schema.json",
            "https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"
         ],
         "properties": {
            "grid:code": "WRS-189-34",
            "proj:code": "EPSG:4326",
            "start_datetime": "2010-12-28T09:31:52Z",
            "end_datetime": "2010-12-28T09:32:21Z",
            "view:sun_azimuth": 156.116,
            "product:type": "TM__GTC_1P",
            "title": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
            "version": "SLA",
            "platform": "Landsat-5",
            "view:sun_elevation": 25.3554,
            "datetime": "2010-12-28T09:31:52Z",
            "instruments": [
               "TM"
            ],
            "constellation": "Landsat",
            "sat:orbit_state": "descending",
            "eo:cloud_cover": 18,
            "eopf:instrument_mode": "IM",
            "updated": "2026-05-29T12:02:51Z",
            "sat:absolute_orbit": 142682,
            "product:acquisition_type": "nominal"
         }
      },
      {
         "stac_version": "1.0.0",
         "assets": {
            "enclosure_zip": {
               "roles": [
                  "data"
               ],
               "href": "https://landsat-diss.eo.esa.int/oads/data/LandsatTM/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP",
               "published": "2010-12-21T09:26:08Z",
               "type": "application/zip",
               "title": "Download"
            },
            "thumbnail": {
               "roles": [
                  "thumbnail"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/thumbnail/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP_TIMG.jpg",
               "type": "image/jpeg",
               "title": "THUMBNAIL"
            },
            "offering_2": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetTile"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-21T09:25:40Z--2010-12-21T09:26:08Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "title": "GetTile",
               "type": "image/jpgpng"
            },
            "quicklook_png": {
               "roles": [
                  "overview"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/browse/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP_BID.PNG",
               "type": "image/png",
               "title": "QUICKLOOK"
            },
            "offering_1": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml",
               "title": "GetCapabilities",
               "type": "application/xml"
            },
            "metadata_ogc_10_157r4": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/gml%2Bxml&recordSchema=om",
               "title": "OGC 10-157r4 metadata",
               "type": "application/gml+xml;profile=\"http://www.opengis.net/spec/EOMPOM/1.1\""
            },
            "offering_4": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetMap"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetMap&layers=Landsat-TM___heatmap&format=image/png&TRANSPARENT=true&TIME=2010-12-21T09:25:40Z/2010-12-21T09:26:08Z&width={Width}&height={Height}&SRS=EPSG:4326&styles={Styles}&bbox={BoundingBox}&dim_range={DimRange}",
               "title": "GetMap",
               "type": "image/png"
            },
            "metadata_ogc_17_003r2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?mode=owc",
               "title": "OGC 17-003r2 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eo-geojson/1.0\""
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "offering_3": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetCapabilities",
               "title": "GetCapabilities",
               "type": "application/xml"
            }
         },
         "bbox": [
            14.1469,
            36.5277,
            16.703,
            38.4169
         ],
         "geometry": {
            "coordinates": [
               [
                  [
                     14.601,
                     38.4169
                  ],
                  [
                     16.703,
                     38.1114
                  ],
                  [
                     16.2047,
                     36.5277
                  ],
                  [
                     14.1469,
                     36.8264
                  ],
                  [
                     14.601,
                     38.4169
                  ]
               ]
            ],
            "type": "Polygon"
         },
         "links": [
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "wms:transparent": true,
               "rel": "wms",
               "wms:dimensions": {
                  "dim_range": "{DimRange}",
                  "TIME": "2010-12-21T09:25:40Z/2010-12-21T09:26:08Z",
                  "version": "1.1.1"
               },
               "href": "https://vs.eo.esa.int/wms07/ows",
               "type": "image/png",
               "title": "LANDSAT.TM.GTC product",
               "wms:layers": [
                  "Landsat-TM___heatmap"
               ],
               "wms:styles": [
                  "{Styles}"
               ]
            },
            {
               "wmts:encoding": "rest",
               "uriTemplate": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-21T09:25:40Z--2010-12-21T09:26:08Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "rel": "wmts",
               "wmts:layer": "Landsat-TM",
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml"
            },
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
               "type": "application/geo+json"
            },
            {
               "rel": "canonical",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
               "type": "application/geo+json"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "collection",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
         "collection": "LANDSAT.TM.GTC",
         "type": "Feature",
         "stac_extensions": [
            "https://stac-extensions.github.io/eo/v2.0.0/schema.json",
            "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
            "https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json",
            "https://stac-extensions.github.io/sat/v1.2.0/schema.json",
            "https://stac-extensions.github.io/view/v1.0.0/schema.json",
            "https://stac-extensions.github.io/product/v1.0.0/schema.json",
            "https://stac-extensions.github.io/grid/v1.1.0/schema.json",
            "https://stac-extensions.github.io/version/v1.2.0/schema.json",
            "https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"
         ],
         "properties": {
            "grid:code": "WRS-188-34",
            "proj:code": "EPSG:4326",
            "start_datetime": "2010-12-21T09:25:40Z",
            "end_datetime": "2010-12-21T09:26:08Z",
            "view:sun_azimuth": 157.001,
            "product:type": "TM__GTC_1P",
            "title": "LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
            "version": "SLA",
            "platform": "Landsat-5",
            "view:sun_elevation": 25.478,
            "datetime": "2010-12-21T09:25:40Z",
            "instruments": [
               "TM"
            ],
            "constellation": "Landsat",
            "sat:orbit_state": "descending",
            "eo:cloud_cover": 7,
            "eopf:instrument_mode": "IM",
            "updated": "2026-05-29T12:02:50Z",
            "sat:absolute_orbit": 142580,
            "product:acquisition_type": "nominal"
         }
      }
   ]
}
```


**Step 5**  
>  Alternatively, search for granules in this collection via the granule search endpoint for the collection advertized in the collection metadata (rel="items").

In [353]:
# Retrieve collection metadata using the collection identifier and extract granule search endpoint
response = requests.get(URL)
data_collections = json.loads(response.text)

expression = parse("$.links[?(@.rel == 'items' )].href")
r = expression.find(data_collections)
r[0].value

'https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items'

In [354]:
args = { 
    'bbox' : '14.9,37.7,14.99,37.78', # Mount Etna
    'datetime': '2010-01-01T00:00:00Z/2011-01-02T00:00:00Z',
    'limit' : '2'  
}

In [355]:
response = requests.get(r[0].value, params=args)
data = json.loads(response.text)
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

```json
{
   "features": [
      {
         "stac_version": "1.0.0",
         "assets": {
            "enclosure_zip": {
               "roles": [
                  "data"
               ],
               "href": "https://landsat-diss.eo.esa.int/oads/data/LandsatTM/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP",
               "published": "2010-12-28T09:32:21Z",
               "type": "application/zip",
               "title": "Download"
            },
            "thumbnail": {
               "roles": [
                  "thumbnail"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/thumbnail/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_TIMG.jpg",
               "type": "image/jpeg",
               "title": "THUMBNAIL"
            },
            "offering_2": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetTile"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "title": "GetTile",
               "type": "image/jpgpng"
            },
            "quicklook_png": {
               "roles": [
                  "overview"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/browse/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_BID.PNG",
               "type": "image/png",
               "title": "QUICKLOOK"
            },
            "offering_1": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml",
               "title": "GetCapabilities",
               "type": "application/xml"
            },
            "metadata_ogc_10_157r4": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/gml%2Bxml&recordSchema=om",
               "title": "OGC 10-157r4 metadata",
               "type": "application/gml+xml;profile=\"http://www.opengis.net/spec/EOMPOM/1.1\""
            },
            "offering_4": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetMap"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetMap&layers=Landsat-TM___heatmap&format=image/png&TRANSPARENT=true&TIME=2010-12-28T09:31:52Z/2010-12-28T09:32:21Z&width={Width}&height={Height}&SRS=EPSG:4326&styles={Styles}&bbox={BoundingBox}&dim_range={DimRange}",
               "title": "GetMap",
               "type": "image/png"
            },
            "metadata_ogc_17_003r2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?mode=owc",
               "title": "OGC 17-003r2 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eo-geojson/1.0\""
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "offering_3": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetCapabilities",
               "title": "GetCapabilities",
               "type": "application/xml"
            }
         },
         "bbox": [
            12.5999,
            36.5266,
            15.1541,
            38.4181
         ],
         "geometry": {
            "coordinates": [
               [
                  [
                     13.0533,
                     38.4181
                  ],
                  [
                     15.1541,
                     38.1131
                  ],
                  [
                     14.6573,
                     36.5266
                  ],
                  [
                     12.5999,
                     36.8248
                  ],
                  [
                     13.0533,
                     38.4181
                  ]
               ]
            ],
            "type": "Polygon"
         },
         "links": [
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "wms:transparent": true,
               "rel": "wms",
               "wms:dimensions": {
                  "dim_range": "{DimRange}",
                  "TIME": "2010-12-28T09:31:52Z/2010-12-28T09:32:21Z",
                  "version": "1.1.1"
               },
               "href": "https://vs.eo.esa.int/wms07/ows",
               "type": "image/png",
               "title": "LANDSAT.TM.GTC product",
               "wms:layers": [
                  "Landsat-TM___heatmap"
               ],
               "wms:styles": [
                  "{Styles}"
               ]
            },
            {
               "wmts:encoding": "rest",
               "uriTemplate": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "rel": "wmts",
               "wmts:layer": "Landsat-TM",
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml"
            },
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "canonical",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "collection",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
         "collection": "LANDSAT.TM.GTC",
         "type": "Feature",
         "stac_extensions": [
            "https://stac-extensions.github.io/eo/v2.0.0/schema.json",
            "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
            "https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json",
            "https://stac-extensions.github.io/sat/v1.2.0/schema.json",
            "https://stac-extensions.github.io/view/v1.0.0/schema.json",
            "https://stac-extensions.github.io/product/v1.0.0/schema.json",
            "https://stac-extensions.github.io/grid/v1.1.0/schema.json",
            "https://stac-extensions.github.io/version/v1.2.0/schema.json",
            "https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"
         ],
         "properties": {
            "grid:code": "WRS-189-34",
            "proj:code": "EPSG:4326",
            "start_datetime": "2010-12-28T09:31:52Z",
            "end_datetime": "2010-12-28T09:32:21Z",
            "view:sun_azimuth": 156.116,
            "product:type": "TM__GTC_1P",
            "title": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
            "version": "SLA",
            "platform": "Landsat-5",
            "view:sun_elevation": 25.3554,
            "datetime": "2010-12-28T09:31:52Z",
            "instruments": [
               "TM"
            ],
            "constellation": "Landsat",
            "sat:orbit_state": "descending",
            "eo:cloud_cover": 18,
            "eopf:instrument_mode": "IM",
            "updated": "2026-05-29T12:02:51Z",
            "sat:absolute_orbit": 142682,
            "product:acquisition_type": "nominal"
         }
      },
      {
         "stac_version": "1.0.0",
         "assets": {
            "enclosure_zip": {
               "roles": [
                  "data"
               ],
               "href": "https://landsat-diss.eo.esa.int/oads/data/LandsatTM/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP",
               "published": "2010-12-21T09:26:08Z",
               "type": "application/zip",
               "title": "Download"
            },
            "thumbnail": {
               "roles": [
                  "thumbnail"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/thumbnail/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP_TIMG.jpg",
               "type": "image/jpeg",
               "title": "THUMBNAIL"
            },
            "offering_2": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetTile"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-21T09:25:40Z--2010-12-21T09:26:08Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "title": "GetTile",
               "type": "image/jpgpng"
            },
            "quicklook_png": {
               "roles": [
                  "overview"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/browse/L05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F_v0100.SIP.ZIP_BID.PNG",
               "type": "image/png",
               "title": "QUICKLOOK"
            },
            "offering_1": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml",
               "title": "GetCapabilities",
               "type": "application/xml"
            },
            "metadata_ogc_10_157r4": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/gml%2Bxml&recordSchema=om",
               "title": "OGC 10-157r4 metadata",
               "type": "application/gml+xml;profile=\"http://www.opengis.net/spec/EOMPOM/1.1\""
            },
            "offering_4": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetMap"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetMap&layers=Landsat-TM___heatmap&format=image/png&TRANSPARENT=true&TIME=2010-12-21T09:25:40Z/2010-12-21T09:26:08Z&width={Width}&height={Height}&SRS=EPSG:4326&styles={Styles}&bbox={BoundingBox}&dim_range={DimRange}",
               "title": "GetMap",
               "type": "image/png"
            },
            "metadata_ogc_17_003r2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?mode=owc",
               "title": "OGC 17-003r2 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eo-geojson/1.0\""
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "offering_3": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetCapabilities",
               "title": "GetCapabilities",
               "type": "application/xml"
            }
         },
         "bbox": [
            14.1469,
            36.5277,
            16.703,
            38.4169
         ],
         "geometry": {
            "coordinates": [
               [
                  [
                     14.601,
                     38.4169
                  ],
                  [
                     16.703,
                     38.1114
                  ],
                  [
                     16.2047,
                     36.5277
                  ],
                  [
                     14.1469,
                     36.8264
                  ],
                  [
                     14.601,
                     38.4169
                  ]
               ]
            ],
            "type": "Polygon"
         },
         "links": [
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "wms:transparent": true,
               "rel": "wms",
               "wms:dimensions": {
                  "dim_range": "{DimRange}",
                  "TIME": "2010-12-21T09:25:40Z/2010-12-21T09:26:08Z",
                  "version": "1.1.1"
               },
               "href": "https://vs.eo.esa.int/wms07/ows",
               "type": "image/png",
               "title": "LANDSAT.TM.GTC product",
               "wms:layers": [
                  "Landsat-TM___heatmap"
               ],
               "wms:styles": [
                  "{Styles}"
               ]
            },
            {
               "wmts:encoding": "rest",
               "uriTemplate": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-21T09:25:40Z--2010-12-21T09:26:08Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "rel": "wmts",
               "wmts:layer": "Landsat-TM",
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml"
            },
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
               "type": "application/geo+json"
            },
            {
               "rel": "canonical",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
               "type": "application/geo+json"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "collection",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
         "collection": "LANDSAT.TM.GTC",
         "type": "Feature",
         "stac_extensions": [
            "https://stac-extensions.github.io/eo/v2.0.0/schema.json",
            "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
            "https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json",
            "https://stac-extensions.github.io/sat/v1.2.0/schema.json",
            "https://stac-extensions.github.io/view/v1.0.0/schema.json",
            "https://stac-extensions.github.io/product/v1.0.0/schema.json",
            "https://stac-extensions.github.io/grid/v1.1.0/schema.json",
            "https://stac-extensions.github.io/version/v1.2.0/schema.json",
            "https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"
         ],
         "properties": {
            "grid:code": "WRS-188-34",
            "proj:code": "EPSG:4326",
            "start_datetime": "2010-12-21T09:25:40Z",
            "end_datetime": "2010-12-21T09:26:08Z",
            "view:sun_azimuth": 157.001,
            "product:type": "TM__GTC_1P",
            "title": "LS05_RMTI_TM__GTC_1P_20101221T092540_20101221T092608_142580_0188_0034_741F",
            "version": "SLA",
            "platform": "Landsat-5",
            "view:sun_elevation": 25.478,
            "datetime": "2010-12-21T09:25:40Z",
            "instruments": [
               "TM"
            ],
            "constellation": "Landsat",
            "sat:orbit_state": "descending",
            "eo:cloud_cover": 7,
            "eopf:instrument_mode": "IM",
            "updated": "2026-05-29T12:02:50Z",
            "sat:absolute_orbit": 142580,
            "product:acquisition_type": "nominal"
         }
      }
   ],
   "numberReturned": 2,
   "links": [
      {
         "rel": "self",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2",
         "type": "application/geo+json"
      },
      {
         "rel": "next",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&pagingInfo=limit=2_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=3",
         "type": "application/geo+json"
      },
      {
         "rel": "last",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&pagingInfo=limit=2_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=31",
         "type": "application/geo+json"
      },
      {
         "rel": "first",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&pagingInfo=limit=2_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=1",
         "type": "application/geo+json"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&httpAccept=application/atom%2Bxml",
         "type": "application/atom+xml"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\""
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2&mode=owc",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/os-geojson/1.0\""
      }
   ],
   "id": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=2",
   "type": "FeatureCollection",
   "numberMatched": 31
}
```


## Obtaining the Queryables Document

A Queryables document provides necessary information for clients to programmatically formulate valid search requests using the `filter` parameter. Specifically, clients are expected to acquire both the name and type of the parameters based on the JSON Schema returned by the `/queryables` endpoint. 

The list of supported filter parameters (a.k.a. queryables) for this collection can be retrieved from the collection metadata as shown below.

In [356]:
# extract the queryables endpoint

expression = parse("$.links[?(@.rel == 'http://www.opengis.net/def/rel/ogc/1.0/queryables' )].href")
r = expression.find(data_collections)
r[0].value

'https://geo.spacebel.be/collections/LANDSAT.TM.GTC/queryables'

At this endpoint, the list of filter parameters is available as a JSON Schema.  The table below shows the filter parameters for this specific collection.

In [357]:
response = requests.get(r[0].value)   
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

,key,type,format
10,datetime,string,-
14,eo:cloud_cover,number,-
16,eopf:instrument_mode,string,-
2,externalId,string,-
21,frame,string,-
15,id,string,-
1,illuminationZenithAngle,number,-
11,instruments,string,-
6,platform,string,-
7,platformSerialIdentifier,string,-


## Search request

The WGISS CDA STAC interface support both searching for collections through the IDN or FedEO and for granules in a specific collection at one of the data partners. It executes a collection or inventory search (a.k.a. granule search), as appropriate, and returns the matching results.

In order to initialize a valid STAC API granule search request, clients should use filter parameters with proper values. The 
available filter parameters for a collection are available in the Queryables response explained in the previous section. 

## Search response

The STAC API returns the collection search responses in JSON format and the granule (i.e. item) search responses in GeoJSON format.

The search response elements include:

- numberReturned: number of results returned in current response page
- numberMatched: number of total results

### Collection search response

In [358]:
# Obtain the collection search endpoint
response = requests.get(URL_LANDING_PAGE)
data = json.loads(response.text)
expression = parse("$.links[?(@.rel == 'data')].href")
r = expression.find(data)
r[0].value

'https://geo.spacebel.be/collections'

Below is an example of a STAC collection search response from the CDA in JSON, to the free keyword search 
“Biomass”.

In [359]:
# Formulate a collection search request
args = { 
    'q' : 'Biomass',
    'limit' : '2'  
}
response = requests.get(r[0].value, params=args)
data = json.loads(response.text)

In [360]:
data['numberReturned']

2

In [361]:
data['numberMatched']

70

In [362]:
# Show complete collection search response
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

```json
{
   "collections": [
      {
         "extent": {
            "spatial": {
               "bbox": [
                  [
                     -180,
                     -90,
                     180,
                     90
                  ]
               ]
            },
            "temporal": {
               "interval": [
                  [
                     "2005-01-01T00:00:00Z",
                     "2024-12-31T23:59:59Z"
                  ]
               ]
            }
         },
         "stac_version": "1.0.0",
         "keywords": [
            "DIF10",
            "ESA",
            "CCI",
            "orthoimagery",
            "EARTH SCIENCE>BIOSPHERE>VEGETATION>BIOMASS",
            "Biomass",
            "P-SAR"
         ],
         "description": "This dataset comprises estimates of forest above-ground biomass (AGB) for the years between 2005 and 2012 and between 2015 and 2024. Examples of AGB change maps have been created for consecutive years (2020-2019) and for a decadal interval (2020-2010). Additionally provided in this version release are aggregated data products. These aggregated products of the AGB and AGB change data layers are available at coarser resolutions (1, 10, 25 and 50km).A Python tool is newly provided to compute AGB change maps for any pair of AGB maps. The AGB maps are derived from a combination of Earth observation data, depending on the year, from the Copernicus Sentinel-1 mission, Envisat\u00e2\u0080\u0099s ASAR (Advanced Synthetic Aperture Radar) instrument and JAXA\u00e2\u0080\u0099s (Japan Aerospace Exploration Agency) Advanced Land Observing Satellite (ALOS-1 and ALOS-2), along with additional information from Earth observation sources, e.g., extended ICESat-2 observations to calibrate retrieval models. A cost function that preserves the temporal features as expressed in the remote sensing data is applied to limit biases between the 2005-2012 and the 2015-2024 AGB maps.The AGB data products consist of two (2) global layers that include estimates of:1) above ground biomass (AGB, unit: tons/ha i.e., Mg/ha) (raster dataset). This is defined as the mass, expressed as oven-dry weight of the woody parts (stem, bark, branches and twigs) of all living trees excluding stump and roots per unit area2) per-pixel estimates of above-ground biomass uncertainty expressed as the standard deviation in Mg/ha (raster dataset)An AGB change product consists of three sets of maps: the AGB change defined as the difference between two AGB maps, the standard deviation of the AGB change and a quality flag of the AGB change.Data are provided in both netcdf and geotiff format.This release of the data is version 7. Compared to version 6, version 7 consists of an update of the maps of AGB for the years 2007, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022 and new AGB maps for 2005, 2006, 2008, 2009, 2011, 2012, 2023 and 2024. The data has been produced as part of the European Space Agency's (ESA's) Climate Change Initiative (CCI) programme by the Biomass CCI team.",
         "collection": "EOP:STFC:CEDA-CCI",
         "type": "Collection",
         "title": "ESA Biomass Climate Change Initiative (Biomass_cci): Global datasets of forest above-ground biomass for the years 2005-2012 and 2015-2024, v7.0",
         "themes": [
            {
               "concepts": [
                  {
                     "id": "1dacc3db-07df-5d93-869a-6bb256f01e51",
                     "title": "P-SAR",
                     "url": "https://earth.esa.int/concept/1dacc3db-07df-5d93-869a-6bb256f01e51"
                  }
               ],
               "scheme": "https://earth.esa.int/concepts/concept_scheme/instruments"
            },
            {
               "concepts": [
                  {
                     "id": "1715936c-434b-410c-aa94-c2f6b5c08c95",
                     "title": "P-SAR",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/1715936c-434b-410c-aa94-c2f6b5c08c95"
                  }
               ],
               "scheme": "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/instruments"
            },
            {
               "concepts": [
                  {
                     "id": "a25a98f2-c03a-51b7-be4a-afab0863186c",
                     "title": "Biomass",
                     "url": "https://earth.esa.int/concept/a25a98f2-c03a-51b7-be4a-afab0863186c"
                  }
               ],
               "scheme": "https://earth.esa.int/concepts/concept_scheme/platforms"
            },
            {
               "concepts": [
                  {
                     "id": "686feba9-87ba-474c-8280-7f67565cfb2f",
                     "title": "EARTH SCIENCE>BIOSPHERE>VEGETATION>BIOMASS",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/686feba9-87ba-474c-8280-7f67565cfb2f"
                  }
               ],
               "scheme": "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/sciencekeywords"
            },
            {
               "concepts": [
                  {
                     "id": "a9b21edd-49b7-43be-8e35-8652bbc5559a",
                     "title": "Biomass",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/a9b21edd-49b7-43be-8e35-8652bbc5559a"
                  }
               ],
               "scheme": "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/platforms"
            }
         ],
         "license": "various",
         "assets": {
            "search": {
               "roles": [
                  "search"
               ],
               "href": "https://fedeo-client.ceos.org?url=https://geo.spacebel.be/api?httpAccept=application/opensearchdescription%252Bxml&uid=6429d1aafe1e43b9b414e4a5a7f8b903",
               "type": "text/html",
               "title": "Search client"
            },
            "enclosure": {
               "roles": [
                  "data"
               ],
               "href": "http://data.ceda.ac.uk/neodc/esacci/biomass/data/agb/maps/v7.0",
               "type": "application/x-binary",
               "title": "DOWNLOAD - Download Data"
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "metadata_iso_19139_2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/vnd.iso.19139-2%2Bxml",
               "title": "ISO 19139-2 metadata",
               "type": "application/vnd.iso.19139-2+xml"
            },
            "metadata_dif_10": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/dif10%2Bxml",
               "title": "DIF-10 metadata",
               "type": "application/dif10+xml"
            },
            "metadata_ogc_17_084r1": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?mode=owc",
               "title": "OGC 17-084r1 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eoc-geojson/1.0\""
            },
            "metadata_iso_191115_3": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/vnd.iso.19115-3%2Bxml",
               "title": "ISO 19115-3 metadata",
               "type": "application/vnd.iso.19115-3+xml"
            }
         },
         "links": [
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903",
               "type": "application/json"
            },
            {
               "rel": "root",
               "href": "https://geo.spacebel.be/",
               "type": "application/json",
               "title": "FEDEO Catalogue"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/",
               "title": "collections",
               "type": "application/json"
            },
            {
               "rel": "items",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903/items",
               "type": "application/geo+json",
               "title": "Datasets search for the series 6429d1aafe1e43b9b414e4a5a7f8b903"
            },
            {
               "rel": "http://www.opengis.net/def/rel/ogc/1.0/queryables",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903/queryables",
               "type": "application/schema+json",
               "title": "Queryables for 6429d1aafe1e43b9b414e4a5a7f8b903"
            },
            {
               "rel": "license",
               "href": "https://artefacts.ceda.ac.uk/licences/specific_licences/esacci_biomass_terms_and_conditions_v2.pdf",
               "type": "application/pdf",
               "title": "Other Constraints"
            },
            {
               "rel": "search",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903/api",
               "type": "application/opensearchdescription+xml",
               "title": "OpenSearch Description Document"
            },
            {
               "rel": "describedby",
               "href": "https://catalogue.ceda.ac.uk/uuid/6429d1aafe1e43b9b414e4a5a7f8b903",
               "title": "CEDA Data Catalogue Page - Detail and access information for the resource"
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/projects/biomass/",
               "title": "ESA CCI Biomass project website - No further details."
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/en/projects/biomass/key-documents/",
               "title": "Project documentation for Biomass CCI - No further details."
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/xml",
               "type": "application/xml",
               "title": "Dublin Core metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/6429d1aafe1e43b9b414e4a5a7f8b903?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            },
            {
               "rel": "related",
               "href": "https://geo.spacebel.be/series/eo:platform/Biomass",
               "title": "More collections for Biomass platform"
            },
            {
               "rel": "related",
               "href": "https://geo.spacebel.be/series/concepts/instruments/1dacc3db-07df-5d93-869a-6bb256f01e51",
               "title": "More collections for P-SAR instrument"
            },
            {
               "rel": "related",
               "href": "https://geo.spacebel.be/series/eo:organisationName/CEDA",
               "title": "More collections for CEDA"
            }
         ],
         "id": "6429d1aafe1e43b9b414e4a5a7f8b903",
         "updated": "2026-10-04T01:14:12Z",
         "stac_extensions": [
            "https://stac-extensions.github.io/themes/v1.0.0/schema.json"
         ],
         "providers": [
            {
               "roles": [
                  "producer"
               ],
               "name": "CEDA"
            },
            {
               "roles": [
                  "producer"
               ],
               "name": "EOP:STFC:CEDA-CCI"
            },
            {
               "roles": [
                  "host"
               ],
               "name": "FEDEO Clearinghouse",
               "url": "https://geo.spacebel.be/"
            }
         ],
         "summaries": {
            "instruments": [
               "P-SAR"
            ],
            "platform": [
               "Biomass"
            ]
         }
      },
      {
         "extent": {
            "spatial": {
               "bbox": [
                  [
                     -180,
                     -90,
                     180,
                     90
                  ]
               ]
            },
            "temporal": {
               "interval": [
                  [
                     "2019-09-01T00:00:00Z",
                     "2025-06-30T23:59:59Z"
                  ]
               ]
            }
         },
         "stac_version": "1.0.0",
         "keywords": [
            "ESA",
            "CCI",
            "Precursors",
            "Ammonia",
            "aerosol precursors",
            "satellite",
            "observation",
            "atmosphere",
            "ammonia",
            "NH3",
            "level 3",
            "total column",
            "agriculture",
            "fertilizers",
            "livestock",
            "biomass burning",
            "industrial emissions",
            "air quality",
            "IASI",
            "orthoimagery",
            "EARTH SCIENCE>OCEANS>OCEAN CHEMISTRY>AMMONIA",
            "EARTH SCIENCE>ATMOSPHERE",
            "EARTH SCIENCE>AGRICULTURE",
            "EARTH SCIENCE>AGRICULTURE>AGRICULTURAL CHEMICALS>FERTILIZERS",
            "EARTH SCIENCE>HUMAN DIMENSIONS>ENVIRONMENTAL IMPACTS>BIOMASS BURNING",
            "EARTH SCIENCE>HUMAN DIMENSIONS>ENVIRONMENTAL IMPACTS>INDUSTRIAL EMISSIONS",
            "EARTH SCIENCE>ATMOSPHERE>AIR QUALITY"
         ],
         "description": "This long-term dataset contains monthly Level 3 (L3) gridded ammonia (NH3) total column data derived from IASI (Infrared Atmospheric Sounding Interferometer) measurements onboard Metop-C (Meteorological operational satellite-C) data produced as part of the European Space Agency's (ESA) Climate Change Initiative (CCI) Precursors for aerosols and ozone project. The temporal range of this dataset is from September 2019 to June 2025. Full documentation can be found at the project website https://climate.esa.int/en/projects/precursors-for-aerosols-and-ozone/.The version number is 4.0.1R. Data are available in NetCDF format.This dataset forms part of the ESA Precursors Climate Change Initiative (Precursors_cci): Precursors IASI NH3 L3 data products v4R.  doi:10.5285/463862326d574951bea52527f9df397d. https://dx.doi.org/10.5285/463862326d574951bea52527f9df397d",
         "collection": "EOP:STFC:CEDA-CCI",
         "type": "Collection",
         "title": "ESA Precursors for Aerosols and Ozone Climate Change Initiative (Precursors_cci): IASI/Metop-C NH3 L3 product, version 4R",
         "themes": [
            {
               "concepts": [
                  {
                     "id": "9d7eed04-9c49-4024-8d0f-06474cc38bbc",
                     "title": "EARTH SCIENCE>HUMAN DIMENSIONS>ENVIRONMENTAL IMPACTS>BIOMASS BURNING",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/9d7eed04-9c49-4024-8d0f-06474cc38bbc"
                  },
                  {
                     "id": "a956d045-3b12-441c-8a18-fac7d33b2b4e",
                     "title": "EARTH SCIENCE>AGRICULTURE",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/a956d045-3b12-441c-8a18-fac7d33b2b4e"
                  },
                  {
                     "id": "07f7ea8e-cf94-4421-923a-539e12dbeb95",
                     "title": "EARTH SCIENCE>HUMAN DIMENSIONS>ENVIRONMENTAL IMPACTS>INDUSTRIAL EMISSIONS",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/07f7ea8e-cf94-4421-923a-539e12dbeb95"
                  },
                  {
                     "id": "77397026-09c9-44e0-b85f-77b2bc9b1630",
                     "title": "EARTH SCIENCE>ATMOSPHERE>AIR QUALITY",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/77397026-09c9-44e0-b85f-77b2bc9b1630"
                  },
                  {
                     "id": "64d17528-29b4-4e2e-843a-7f7035bb5717",
                     "title": "EARTH SCIENCE>OCEANS>OCEAN CHEMISTRY>AMMONIA",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/64d17528-29b4-4e2e-843a-7f7035bb5717"
                  },
                  {
                     "id": "c47f6052-634e-40ef-a5ac-13f69f6f4c2a",
                     "title": "EARTH SCIENCE>ATMOSPHERE",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/c47f6052-634e-40ef-a5ac-13f69f6f4c2a"
                  },
                  {
                     "id": "18a8197e-3a3f-408c-9c51-e9fe89dd6b45",
                     "title": "EARTH SCIENCE>AGRICULTURE>AGRICULTURAL CHEMICALS>FERTILIZERS",
                     "url": "https://gcmd.earthdata.nasa.gov/kms/concept/18a8197e-3a3f-408c-9c51-e9fe89dd6b45"
                  }
               ],
               "scheme": "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/sciencekeywords"
            }
         ],
         "license": "various",
         "assets": {
            "enclosure": {
               "roles": [
                  "data"
               ],
               "href": "http://data.ceda.ac.uk/neodc/esacci/precursors/data/IASI_NH3_L3/v4R/METOP-C",
               "type": "application/x-binary",
               "title": "DOWNLOAD - Download Data"
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "metadata_iso_19139_2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/vnd.iso.19139-2%2Bxml",
               "title": "ISO 19139-2 metadata",
               "type": "application/vnd.iso.19139-2+xml"
            },
            "metadata_dif_10": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/dif10%2Bxml",
               "title": "DIF-10 metadata",
               "type": "application/dif10+xml"
            },
            "metadata_ogc_17_084r1": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?mode=owc",
               "title": "OGC 17-084r1 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eoc-geojson/1.0\""
            },
            "metadata_iso_191115_3": {
               "roles": [
                  "metadata"
               ],
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/vnd.iso.19115-3%2Bxml",
               "title": "ISO 19115-3 metadata",
               "type": "application/vnd.iso.19115-3+xml"
            }
         },
         "links": [
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4",
               "type": "application/json"
            },
            {
               "rel": "root",
               "href": "https://geo.spacebel.be/",
               "type": "application/json",
               "title": "FEDEO Catalogue"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/",
               "title": "collections",
               "type": "application/json"
            },
            {
               "rel": "license",
               "href": "https://artefacts.ceda.ac.uk/licences/specific_licences/esacci_precursors_for_aerosols_and_ozone_terms_and_conditions.pdf",
               "type": "application/pdf",
               "title": "Other Constraints"
            },
            {
               "rel": "describedby",
               "href": "https://catalogue.ceda.ac.uk/uuid/c1cee19b22ae4c6187605ada036477e4",
               "title": "CEDA Data Catalogue Page - Detail and access information for the resource"
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/documents/2636/Precursors_cci_D4.2_PUG_01_01.pdf",
               "type": "application/pdf",
               "title": "Product User Guide - No further details."
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/en/projects/precursors-for-aerosols-and-ozone/",
               "title": "ESA CCI Precursors Project Website - No further details."
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/en/#/",
               "title": "ESA Climate Change Initiative Website - No further details."
            },
            {
               "rel": "describedby",
               "href": "https://climate.esa.int/documents/2632/Precursors_cci_D2.1_ATBD_01_01.pdf",
               "type": "application/pdf",
               "title": "Algorithm Theoretical Basis Document - No further details."
            },
            {
               "rel": "describedby",
               "href": "https://amt.copernicus.org/articles/16/5009/2023/",
               "title": "Clarisse, L., Franco, B., Van Damme, M., Di Gioacchino, T., Hadji-Lazaro, J., Whitburn, S., Noppen, L., Hurtmans, D., Clerbaux, C., and Coheur, P.: The IASI NH3 version 4 product: averaging kernels and improved consistency, Atmos. Meas. Tech., 16, 5009\u00e2\u0080\u00935028, doi: 10.5194/amt-16-5009-2023, 2023. - No further details."
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/xml",
               "type": "application/xml",
               "title": "Dublin Core metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/c1cee19b22ae4c6187605ada036477e4?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "c1cee19b22ae4c6187605ada036477e4",
         "updated": "2026-10-04T01:14:33Z",
         "stac_extensions": [
            "https://stac-extensions.github.io/themes/v1.0.0/schema.json"
         ],
         "providers": [
            {
               "roles": [
                  "producer"
               ],
               "name": "CEDA"
            },
            {
               "roles": [
                  "producer"
               ],
               "name": "EOP:STFC:CEDA-CCI"
            },
            {
               "roles": [
                  "host"
               ],
               "name": "FEDEO Clearinghouse",
               "url": "https://geo.spacebel.be/"
            }
         ]
      }
   ],
   "numberReturned": 2,
   "links": [
      {
         "rel": "self",
         "href": "https://geo.spacebel.be/collections?q=Biomass&limit=2",
         "type": "application/json",
         "title": "This document"
      },
      {
         "rel": "next",
         "href": "https://geo.spacebel.be/collections?q=Biomass&limit=2&startRecord=3",
         "type": "application/json",
         "title": "Next results"
      },
      {
         "rel": "http://www.opengis.net/def/rel/ogc/1.0/queryables",
         "href": "https://geo.spacebel.be/collections/queryables",
         "type": "application/schema+json",
         "title": "Queryables for collection search"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections?q=Biomass&limit=2&httpAccept=application/atom%2Bxml",
         "type": "application/atom+xml"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections?q=Biomass&limit=2&httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\""
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections?q=Biomass&limit=2&mode=owc",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/os-geojson/1.0\""
      }
   ],
   "numberMatched": 70
}
```


#### Collection assets

In [363]:
# Get first collection from above response
# Use stac_client class for STAC collection
c = Collection.from_dict(data['collections'][0])
# print("id\t\t:", c.id)
# print("title\t\t:", c.title)
assets = c.assets
df = pd.DataFrame(columns=['roles', 'title', 'type'])
for key in assets:
    ndf = pd.DataFrame({ 
            'roles': assets[key].roles, 
            'type': assets[key].media_type, 
            'title': assets[key].title, 
            # 'href': assets[key].href  
        }, index = [0])
    df = pd.concat([df, ndf], ignore_index=True)
df

,roles,title,type
0,search,Search client,text/html
1,data,DOWNLOAD - Download Data,application/x-binary
2,metadata,ISO 19139 metadata,application/vnd.iso.19139+xml
3,metadata,ISO 19139-2 metadata,application/vnd.iso.19139-2+xml
4,metadata,DIF-10 metadata,application/dif10+xml
5,metadata,OGC 17-084r1 metadata,"application/geo+json;profile=""http://www.openg..."
6,metadata,ISO 19115-3 metadata,application/vnd.iso.19115-3+xml


#### Collection links

Collections provide access to additional resources via `links`.  The `rel` attribute indicates the purpose of the resource. The `href` attribute provides the URL to access the resource.  Collection assets may include `thumbnail` (when available), `search` interfaces, and various `metadata` formats. 

In [364]:
# Display links belonging to the collection
links = c.links
df = pd.DataFrame(columns=['rel', 'title', 'type'])
for link in links:    
    ndf = pd.DataFrame({ 'rel': link.rel,'type': link.media_type, 'title': link.title }, index = [0])
    df = pd.concat([df, ndf], ignore_index=True)
df

,rel,title,type
0,self,None,application/json
1,root,FEDEO Catalogue,application/json
2,parent,collections,application/json
3,items,Datasets search for the series 6429d1aafe1e43b...,application/geo+json
4,http://www.opengis.net/def/rel/ogc/1.0/queryables,Queryables for 6429d1aafe1e43b9b414e4a5a7f8b903,application/schema+json
5,license,Other Constraints,application/pdf
6,search,OpenSearch Description Document,application/opensearchdescription+xml
7,describedby,CEDA Data Catalogue Page - Detail and access i...,None
8,describedby,ESA CCI Biomass project website - No further d...,None
9,describedby,Project documentation for Biomass CCI - No fur...,None


### Granule search response



In [365]:
# Retrieve collection metadata using the collection identifier and extract granule search endpoint
URL = URL_LANDING_PAGE + 'collections/' + COLLECTION_ID2

response = requests.get(URL)
data_collections = json.loads(response.text)

expression = parse("$.links[?(@.rel == 'items' )].href")
r = expression.find(data_collections)
r[0].value

'https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items'

Below is an example of a STAC granule search response from the CDA in GeoJSON.

In [366]:
args = { 
    'bbox' : '14.9,37.7,14.99,37.78', # Mount Etna
    'datetime': '2010-01-01T00:00:00Z/2011-01-02T00:00:00Z',
    'limit' : '1'  
}

In [367]:
response = requests.get(r[0].value, params=args)
data = json.loads(response.text)

In [368]:
data['numberReturned']

1

In [369]:
data['numberMatched']

31

In [370]:
# Show complete granule search response
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

```json
{
   "features": [
      {
         "stac_version": "1.0.0",
         "assets": {
            "enclosure_zip": {
               "roles": [
                  "data"
               ],
               "href": "https://landsat-diss.eo.esa.int/oads/data/LandsatTM/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP",
               "published": "2010-12-28T09:32:21Z",
               "type": "application/zip",
               "title": "Download"
            },
            "thumbnail": {
               "roles": [
                  "thumbnail"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/thumbnail/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_TIMG.jpg",
               "type": "image/jpeg",
               "title": "THUMBNAIL"
            },
            "offering_2": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetTile"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "title": "GetTile",
               "type": "image/jpgpng"
            },
            "quicklook_png": {
               "roles": [
                  "overview"
               ],
               "href": "http://landsat-diss.eo.esa.int/oads/meta/LandsatTM/browse/L05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C_v0100.SIP.ZIP_BID.PNG",
               "type": "image/png",
               "title": "QUICKLOOK"
            },
            "offering_1": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wmts#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml",
               "title": "GetCapabilities",
               "type": "application/xml"
            },
            "metadata_ogc_10_157r4": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/gml%2Bxml&recordSchema=om",
               "title": "OGC 10-157r4 metadata",
               "type": "application/gml+xml;profile=\"http://www.opengis.net/spec/EOMPOM/1.1\""
            },
            "offering_4": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetMap"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetMap&layers=Landsat-TM___heatmap&format=image/png&TRANSPARENT=true&TIME=2010-12-28T09:31:52Z/2010-12-28T09:32:21Z&width={Width}&height={Height}&SRS=EPSG:4326&styles={Styles}&bbox={BoundingBox}&dim_range={DimRange}",
               "title": "GetMap",
               "type": "image/png"
            },
            "metadata_ogc_17_003r2": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?mode=owc",
               "title": "OGC 17-003r2 metadata",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/eo-geojson/1.0\""
            },
            "metadata_iso_19139": {
               "roles": [
                  "metadata"
               ],
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/vnd.iso.19139%2Bxml",
               "title": "ISO 19139 metadata",
               "type": "application/vnd.iso.19139+xml"
            },
            "offering_3": {
               "roles": [
                  "http://www.opengis.net/spec/owc-geojson/1.0/req/wms#GetCapabilities"
               ],
               "href": "https://vs.eo.esa.int/wms07/ows?service=WMS&version=1.1.1&request=GetCapabilities",
               "title": "GetCapabilities",
               "type": "application/xml"
            }
         },
         "bbox": [
            12.5999,
            36.5266,
            15.1541,
            38.4181
         ],
         "geometry": {
            "coordinates": [
               [
                  [
                     13.0533,
                     38.4181
                  ],
                  [
                     15.1541,
                     38.1131
                  ],
                  [
                     14.6573,
                     36.5266
                  ],
                  [
                     12.5999,
                     36.8248
                  ],
                  [
                     13.0533,
                     38.4181
                  ]
               ]
            ],
            "type": "Polygon"
         },
         "links": [
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "wms:transparent": true,
               "rel": "wms",
               "wms:dimensions": {
                  "dim_range": "{DimRange}",
                  "TIME": "2010-12-28T09:31:52Z/2010-12-28T09:32:21Z",
                  "version": "1.1.1"
               },
               "href": "https://vs.eo.esa.int/wms07/ows",
               "type": "image/png",
               "title": "LANDSAT.TM.GTC product",
               "wms:layers": [
                  "Landsat-TM___heatmap"
               ],
               "wms:styles": [
                  "{Styles}"
               ]
            },
            {
               "wmts:encoding": "rest",
               "uriTemplate": "https://vs.eo.esa.int/wmts07/1.0.0/Landsat-TM/default/2010-12-28T09:31:52Z--2010-12-28T09:32:21Z/WGS84/{TileMatrix}/{TileRow}/{TileCol}.jpgpng",
               "rel": "wmts",
               "wmts:layer": "Landsat-TM",
               "href": "https://vs.eo.esa.int/wmts07/1.0.0/WMTSCapabilities.xml"
            },
            {
               "rel": "self",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "canonical",
               "href": "https://catalog.maap.eo.esa.int/catalogue/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
               "type": "application/geo+json"
            },
            {
               "rel": "parent",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "collection",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC",
               "type": "application/json"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/atom%2Bxml",
               "type": "application/atom+xml",
               "title": "Atom format"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
               "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\"",
               "title": "OGC 17-069r3 metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson;profile=https://schema.org",
               "type": "application/ld+json;profile=\"https://schema.org\"",
               "title": "JSON-LD (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/ld%2Bjson",
               "type": "application/ld+json;profile=\"http://data.europa.eu/930/\"",
               "title": "JSON-LD (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml;profile=https://schema.org",
               "type": "application/rdf+xml;profile=\"https://schema.org\"",
               "title": "RDF/XML (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=application/rdf%2Bxml",
               "type": "application/rdf+xml;profile=\"http://data.europa.eu/930/\"",
               "title": "RDF/XML (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle;profile=https://schema.org",
               "type": "text/turtle;profile=\"https://schema.org\"",
               "title": "Turtle (schema.org) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/turtle",
               "type": "text/turtle;profile=\"http://data.europa.eu/930/\"",
               "title": "Turtle (GeoDCAT-AP) metadata"
            },
            {
               "rel": "alternate",
               "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items/LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C?httpAccept=text/html",
               "type": "text/html",
               "title": "HTML"
            }
         ],
         "id": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
         "collection": "LANDSAT.TM.GTC",
         "type": "Feature",
         "stac_extensions": [
            "https://stac-extensions.github.io/eo/v2.0.0/schema.json",
            "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
            "https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json",
            "https://stac-extensions.github.io/sat/v1.2.0/schema.json",
            "https://stac-extensions.github.io/view/v1.0.0/schema.json",
            "https://stac-extensions.github.io/product/v1.0.0/schema.json",
            "https://stac-extensions.github.io/grid/v1.1.0/schema.json",
            "https://stac-extensions.github.io/version/v1.2.0/schema.json",
            "https://stac-extensions.github.io/web-map-links/v1.2.0/schema.json"
         ],
         "properties": {
            "grid:code": "WRS-189-34",
            "proj:code": "EPSG:4326",
            "start_datetime": "2010-12-28T09:31:52Z",
            "end_datetime": "2010-12-28T09:32:21Z",
            "view:sun_azimuth": 156.116,
            "product:type": "TM__GTC_1P",
            "title": "LS05_RMTI_TM__GTC_1P_20101228T093152_20101228T093221_142682_0189_0034_0F4C",
            "version": "SLA",
            "platform": "Landsat-5",
            "view:sun_elevation": 25.3554,
            "datetime": "2010-12-28T09:31:52Z",
            "instruments": [
               "TM"
            ],
            "constellation": "Landsat",
            "sat:orbit_state": "descending",
            "eo:cloud_cover": 18,
            "eopf:instrument_mode": "IM",
            "updated": "2026-05-29T12:02:51Z",
            "sat:absolute_orbit": 142682,
            "product:acquisition_type": "nominal"
         }
      }
   ],
   "numberReturned": 1,
   "links": [
      {
         "rel": "self",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1",
         "type": "application/geo+json"
      },
      {
         "rel": "next",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&pagingInfo=limit=1_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=2",
         "type": "application/geo+json"
      },
      {
         "rel": "last",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&pagingInfo=limit=1_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=31",
         "type": "application/geo+json"
      },
      {
         "rel": "first",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&pagingInfo=limit=1_______bbox=14.9%2C37.7%2C14.99%2C37.78_______datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z_______startRecord=1",
         "type": "application/geo+json"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&httpAccept=application/atom%2Bxml",
         "type": "application/atom+xml"
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&httpAccept=application/geo%2Bjson;profile=http://www.opengis.net/spec/ogcapi-features-1/1.0",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/ogcapi-features-1/1.0\""
      },
      {
         "rel": "alternate",
         "href": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1&mode=owc",
         "type": "application/geo+json;profile=\"http://www.opengis.net/spec/os-geojson/1.0\""
      }
   ],
   "id": "https://geo.spacebel.be/collections/LANDSAT.TM.GTC/items?bbox=14.9%2C37.7%2C14.99%2C37.78&datetime=2010-01-01T00%3A00%3A00Z%2F2011-01-02T00%3A00%3A00Z&limit=1",
   "type": "FeatureCollection",
   "numberMatched": 31
}
```


#### Granule assets

In [371]:
# Show assets of first search result (GeoJSON)
df = pd.DataFrame(columns=['roles', 'title', 'type'])
    
# Display assets belonging to first item in results
for item in granule_results.items():
    assets = item.assets
    for key in assets: 
        try: 
            ndf = pd.DataFrame({ 
                'roles': str(assets[key].roles), 
                'type': assets[key].media_type, 
                'title': assets[key].title, 
                # 'href': assets[key].href  
            }, index = [0])
            df = pd.concat([df, ndf], ignore_index=True)
        except:
            pass
    # stop after first item
    break
df

,roles,title,type
0,['data'],Download,application/zip
1,['thumbnail'],THUMBNAIL,image/jpeg
2,['http://www.opengis.net/spec/owc-geojson/1.0/...,GetTile,image/jpgpng
3,['overview'],QUICKLOOK,image/png
4,['http://www.opengis.net/spec/owc-geojson/1.0/...,GetCapabilities,application/xml
5,['metadata'],OGC 10-157r4 metadata,"application/gml+xml;profile=""http://www.opengi..."
6,['http://www.opengis.net/spec/owc-geojson/1.0/...,GetMap,image/png
7,['metadata'],OGC 17-003r2 metadata,"application/geo+json;profile=""http://www.openg..."
8,['metadata'],ISO 19139 metadata,application/vnd.iso.19139+xml
9,['http://www.opengis.net/spec/owc-geojson/1.0/...,GetCapabilities,application/xml


#### Granule links

In [372]:
# Show assets of first search result (GeoJSON)
df = pd.DataFrame(columns=['rel', 'title', 'type'])
    
# Display assets belonging to first item in results
for item in granule_results.items():
    links = item.links
    for link in links:    
        ndf = pd.DataFrame({ 'rel': link.rel,'type': link.media_type, 'title': link.title }, index = [0])
        df = pd.concat([df, ndf], ignore_index=True)
    # stop after first item
    break
df

,rel,title,type
0,alternate,JSON-LD (schema.org) metadata,"application/ld+json;profile=""https://schema.org"""
1,alternate,JSON-LD (GeoDCAT-AP) metadata,"application/ld+json;profile=""http://data.europ..."
2,wms,LANDSAT.TM.GTC product,image/png
3,wmts,None,None
4,self,None,application/geo+json
5,canonical,None,application/geo+json
6,parent,None,application/json
7,collection,None,application/json
8,alternate,Atom format,application/atom+xml
9,alternate,OGC 17-069r3 metadata,"application/geo+json;profile=""http://www.openg..."
